# ChefBot — фінальний AI-проєкт

ChefBot допомагає домашньому кухарю знайти страву з наявних продуктів,
конвертувати кулінарні одиниці та підібрати заміни інгредієнтів.

Цей ноутбук є відтворюваним сценарієм запуску. Дані, tools, prompt,
агент і evaluation зберігаються окремими модулями у GitHub.

## 1. Завантаження проєкту

Публічний репозиторій не потребує GitHub-токена. Повторний запуск
оновлює наявну чисту копію через fast-forward.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/delef/o-ai.git"
PROJECT_DIR = Path("/content/o-ai")

if PROJECT_DIR.exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

os.chdir(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt"],
    check=True,
)
print("Проєкт і залежності готові.")

Проєкт і залежності готові.


## 2. API-ключ

Додайте `OPENAI_API_KEY` у Colab → **Secrets**. Значення ключа не
друкується та не записується у ноутбук.

In [2]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Додайте OPENAI_API_KEY у Colab Secrets.")
print("OPENAI_API_KEY завантажено з Colab Secrets.")

OPENAI_API_KEY завантажено з Colab Secrets.


## 3. Детерміновані тести без витрат API

In [3]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## 4. Створення ChefBot

In [4]:
from langchain.messages import HumanMessage
from chefbot import create_chefbot, run_chefbot

agent = create_chefbot(api_key=api_key)
print("ChefBot готовий.")

ChefBot готовий.


## 5. Перевірка ключових сценаріїв

Нижче показано не лише фінальну відповідь, а й фактичний tool routing.

In [5]:
DEMO_QUERIES = [
    "Що приготувати з курки та картоплі?",
    "Скільки грамів у 2 склянках борошна?",
    "Чим замінити яйця у випічці?",
]

for query in DEMO_QUERIES:
    result = run_chefbot(agent, [HumanMessage(content=query)])
    print(f"\nКористувач: {query}")
    print("Tools:", [f"{event.name}:{event.status}" for event in result.tool_events])
    print("ChefBot:", result.answer)


Користувач: Що приготувати з курки та картоплі?
Tools: ['recipe_search:ok']
ChefBot: Ось два смачні рецепти з курки та картоплі:

### 1. Курка з картоплею
**Інгредієнти:**
- Куряче філе — 500 г
- Картопля — 700 г
- Морква — 150 г
- Олія — 2 ст.л.
- Сіль — за смаком
- Перець — за смаком

**Час:** 55 хвилин  
**Порцій:** 4

**Приготування:**
1. Наріжте курку, картоплю та моркву.
2. Додайте олію, сіль і перець.
3. Запікайте при 190°C приблизно 40–45 хвилин.

---

### 2. Вареники з картоплею
**Інгредієнти:**
- Борошно — 500 г
- Вода — 250 мл
- Яйця — 1 шт.
- Картопля — 700 г
- Цибуля — 150 г
- Олія — 1 ст.л.
- Сіль — за смаком

**Час:** 80 хвилин  
**Порцій:** 5

**Приготування:**
1. Замісіть тісто з борошна, води, яйця та солі.
2. Приготуйте начинку з картоплі й цибулі.
3. Сформуйте вареники.
4. Варіть у підсоленій воді 4–5 хвилин після спливання.

Обирайте, що вам більше до вподоби!

Користувач: Скільки грамів у 2 склянках борошна?
Tools: ['unit_converter:ok']
ChefBot: У 2 склянках боро

## 6. Контекст розмови та цілісне оновлення рецепта

Другий запит використовує історію першого, тому користувачеві не треба
повторювати назву рецепта. `recipe_editor` повертає повний оновлений
склад і всі кроки, а не частковий патч.

In [6]:
messages = [HumanMessage(content="Знайди рецепт курки з картоплею.")]
first = run_chefbot(agent, messages)
messages = [
    *first.messages,
    HumanMessage(content="Додай до рецепта 1 цибулину і онови весь рецепт."),
]
second = run_chefbot(agent, messages)
revisions = [
    event.artifact
    for event in second.tool_events
    if event.name == "recipe_editor" and event.status == "ok"
]

print("Перший turn tools:", [event.name for event in first.tool_events])
print("Другий turn tools:", [event.name for event in second.tool_events])
if not revisions:
    raise RuntimeError("recipe_editor не повернув цілісне оновлення рецепта.")
revision = revisions[-1]
print("Оновлені інгредієнти:", revision["ingredients"])
print("Оновлені кроки:", revision["steps"])

Перший turn tools: ['recipe_search']
Другий turn tools: ['recipe_editor']
Оновлені інгредієнти: [{'name': 'Куряче філе', 'quantity': 500.0, 'unit': 'г', 'note': None}, {'name': 'Картопля', 'quantity': 700.0, 'unit': 'г', 'note': None}, {'name': 'Морква', 'quantity': 150.0, 'unit': 'г', 'note': None}, {'name': 'Цибуля', 'quantity': 1.0, 'unit': 'шт.', 'note': None}, {'name': 'Олія', 'quantity': 2.0, 'unit': 'ст.л.', 'note': None}, {'name': 'Сіль', 'quantity': None, 'unit': None, 'note': 'за смаком'}, {'name': 'Перець', 'quantity': None, 'unit': None, 'note': 'за смаком'}]
Оновлені кроки: ['Наріжте курку, картоплю, моркву та цибулю.', 'Додайте олію, сіль і перець.', 'Запікайте при 190°C приблизно 40–45 хвилин.']


## 7. Повний evaluation

Результат містить pass/fail, routing, latency, токени та орієнтовну
вартість. CSV можна використати для подальшого аналізу системи.

In [8]:
from chefbot.evaluation import (
    DEFAULT_OUTPUT,
    load_scenarios,
    run_evaluation,
    write_results,
)

scenarios = load_scenarios()
rows = run_evaluation(agent, scenarios, "gpt-4o-mini")
write_results(rows, DEFAULT_OUTPUT)

passed = sum(1 for row in rows if row["passed"])
print(f"Evaluation: {passed}/{len(rows)} passed")
from IPython.display import display
import pandas as pd

display(
    pd.DataFrame(rows)[
        [
            "scenario_id",
            "passed",
            "observed_tools",
            "latency_ms",
            "total_tokens",
            "estimated_cost_usd",
            "failure_reason",
        ]
    ]
)
print("CSV:", DEFAULT_OUTPUT)

def download_results():
    from google.colab import files

    files.download(str(DEFAULT_OUTPUT))


print("Для завантаження CSV виконайте: download_results()")

Evaluation: 14/14 passed


,scenario_id,passed,observed_tools,latency_ms,total_tokens,estimated_cost_usd,failure_reason
0,recipe_exact,True,recipe_search,4244,3741,0.000420,
1,recipe_by_ingredients,True,recipe_search,5679,4240,0.000536,
2,recipe_by_category,True,recipe_search,7113,4154,0.000534,
3,gluten_free,True,recipe_search,5977,4297,0.000602,
4,conversion_direct,True,unit_converter,1283,3424,0.000296,
5,conversion_unsupported,True,unit_converter,1991,3478,0.000438,
6,substitution_known,True,substitution_finder,4038,3749,0.000531,
7,combined_tools,True,"recipe_search,substitution_finder",5268,4456,0.000545,
8,context_follow_up,True,"recipe_search,substitution_finder",7306,8071,0.000827,
9,recipe_revision,True,"recipe_search,recipe_editor",9961,9498,0.001074,


CSV: /content/o-ai/evaluation/results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Для завантаження CSV виконайте: download_results()


## 8. Межі рішення

- ChefBot не вигадує точні рецепти, конвертації або заміни поза локальною базою.
- Серйозні алергії потребують перевірки маркування конкретного продукту.
- Поточна структурована база не потребує vector database або кількох агентів.
- Веб-інтерфейс запускається з кореня репозиторію командою
  `streamlit run app.py`.